# Publishable Model Idea: Causal Workflow-Conditioned BariatricRSD

This notebook updates the BariatricRSD idea into a version that is much easier to defend in a paper.

The old story was: use an offline phase-order cluster as a token. That is interesting, but it risks leakage because a full-video phase order is not known early in surgery.

The publishable story is:

**Remaining surgery duration improves when the model estimates the current workflow style from the observed surgical prefix and uses that prefix-only workflow belief to condition duration, deviation, and phase prediction.**

This turns the phase-order token from an oracle label into a causal, online, testable mechanism.


## 1. The Pivot

Current idea:

- Full video -> phase sequence -> k-means cluster -> token -> RSD prediction.
- Problem: the cluster can encode future phase order.

Publishable idea:

- Observed prefix only -> workflow-state posterior -> soft style token -> causal temporal model -> calibrated RSD distribution.
- Full-video cluster is kept only as an **oracle upper bound**.

This is the difference between a clever ablation and a publishable online surgical AI method.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Patch
from matplotlib.lines import Line2D

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent

LABEL_DIR = PROJECT / 'lambda_mirror' / 'labels'
LOG_DIR = PROJECT / 'lambda_mirror' / 'logs'
FIG_DIR = PROJECT / 'notebooks' / 'method_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 250,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COL = {
    'input': '#dbeafe',
    'visual': '#bbf7d0',
    'temporal': '#fde68a',
    'workflow': '#ddd6fe',
    'head': '#fecdd3',
    'loss': '#f5f5f5',
    'edge': '#27272a',
    'bad': '#fee2e2',
    'good': '#dcfce7',
}

def box(ax, x, y, w, h, text, color, fs=9, lw=1.2):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle='round,pad=0.035,rounding_size=0.08',
        linewidth=lw,
        edgecolor=COL['edge'],
        facecolor=color,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=fs)
    return patch

def arrow(ax, x1, y1, x2, y2, label=None, fs=8):
    arr = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='->', mutation_scale=13, linewidth=1.1, color=COL['edge'])
    ax.add_patch(arr)
    if label:
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.08, label, ha='center', fontsize=fs, color='#52525b')
    return arr

print('ready')


## 2. Publishable Claim

A strong claim should be narrow enough to test and broad enough to matter:

> **Workflow-variable RSD prediction benefits from causal workflow-state conditioning.**

Definitions:

- **Causal**: only frames and labels observed up to time `t` are used.
- **Workflow-state**: a probability distribution over operative styles/orderings inferred from the prefix.
- **Conditioning**: the posterior is converted into a soft token or FiLM vector that modulates temporal features.
- **RSD prediction**: predict remaining minutes and uncertainty, not just a normalized scalar.

This avoids claiming that the model magically knows the final phase order at the start of the case.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('Old Idea vs Publishable Idea', fontsize=13, pad=10)

box(ax, 0.4, 3.5, 2.0, 0.7, 'Full video\nall phases', COL['bad'])
box(ax, 3.0, 3.5, 2.0, 0.7, 'Offline cluster\nfrom full order', COL['bad'])
box(ax, 5.6, 3.5, 2.0, 0.7, 'Phase-order\ntoken', COL['workflow'])
box(ax, 8.2, 3.5, 2.0, 0.7, 'RSD model', COL['temporal'])
box(ax, 10.8, 3.5, 2.2, 0.7, 'Risk:\nfuture leakage', COL['bad'])
for x in [2.4, 5.0, 7.6, 10.2]:
    arrow(ax, x, 3.85, x + 0.55, 3.85)
ax.text(0.4, 4.55, 'Old framing: useful ablation, weak for online publication', fontsize=10, color='#991b1b')

box(ax, 0.4, 1.25, 2.0, 0.7, 'Observed prefix\nframes <= t', COL['input'])
box(ax, 3.0, 1.25, 2.0, 0.7, 'Prefix workflow\nposterior q(z|prefix)', COL['workflow'])
box(ax, 5.6, 1.25, 2.0, 0.7, 'Soft style token\nE[q(z)]', COL['workflow'])
box(ax, 8.2, 1.25, 2.0, 0.7, 'Causal temporal\nmodel', COL['temporal'])
box(ax, 10.8, 1.25, 2.2, 0.7, 'RSD + uncertainty\ncurrent time t', COL['good'])
for x in [2.4, 5.0, 7.6, 10.2]:
    arrow(ax, x, 1.6, x + 0.55, 1.6)
ax.text(0.4, 2.3, 'New framing: online, leakage-safe, experimentally testable', fontsize=10, color='#166534')

fig.tight_layout()
fig.savefig(FIG_DIR / 'old_vs_publishable.png', bbox_inches='tight')
plt.show()


## 3. Proposed Model: CW-BariatricRSD

Name: **Causal Workflow-Conditioned BariatricRSD**.

Inputs at time `t`:

- sampled frame prefix `x_1, ..., x_t`
- optional non-future metadata: center, surgeon ID, procedure variant, preoperative plan
- observed prefix phase predictions or phase labels during training

Outputs at time `t`:

- remaining duration distribution: mean, quantiles, or Gaussian/log-normal parameters
- deviation probability
- current phase probability
- optional progress/completion probability

Key rule: **nothing from frames or labels after `t` can enter the model input.**


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6.2))
ax.set_xlim(0, 15)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('CW-BariatricRSD: Causal Workflow-Conditioned Architecture', fontsize=13, pad=10)

for i in range(6):
    box(ax, 0.5 + i * 0.55, 5.6, 0.42, 0.55, f't-{5-i}', COL['input'], fs=7)
box(ax, 4.1, 5.45, 1.4, 0.85, 'current\ntime t', '#bfdbfe', fs=8)
box(ax, 0.6, 4.35, 4.9, 0.65, 'Visual encoder: ViT / surgical foundation encoder', COL['visual'], fs=9)
box(ax, 0.6, 3.15, 4.9, 0.7, 'Causal temporal encoder\nmasked HTA / causal Transformer', COL['temporal'], fs=9)
for x in [1.0, 1.55, 2.1, 2.65, 3.2, 3.75, 4.8]:
    arrow(ax, x, 5.45, x, 5.0)
arrow(ax, 3.0, 4.35, 3.0, 3.85)

box(ax, 6.4, 5.45, 2.5, 0.85, 'Prefix workflow features\nphase histogram + transitions\nelapsed time + metadata', COL['input'], fs=8)
box(ax, 6.4, 4.15, 2.5, 0.85, 'Workflow posterior net\nq(z_t | prefix)', COL['workflow'], fs=9)
box(ax, 6.4, 2.95, 2.5, 0.75, 'Soft style token\ns_t = sum_k q_k E_k', COL['workflow'], fs=9)
arrow(ax, 7.65, 5.45, 7.65, 5.0)
arrow(ax, 7.65, 4.15, 7.65, 3.7)

box(ax, 9.9, 3.35, 2.0, 0.85, 'Fusion\nattention / FiLM\n/ gated add', '#e9d5ff', fs=9)
arrow(ax, 5.5, 3.5, 9.9, 3.75, 'temporal state')
arrow(ax, 8.9, 3.33, 9.9, 3.75, 'style token')

box(ax, 12.7, 4.7, 1.7, 0.65, 'RSD\ndistribution', COL['head'], fs=8)
box(ax, 12.7, 3.65, 1.7, 0.65, 'Deviation\nprobability', COL['head'], fs=8)
box(ax, 12.7, 2.6, 1.7, 0.65, 'Current phase\nprobability', COL['head'], fs=8)
box(ax, 12.7, 1.55, 1.7, 0.65, 'Progress /\ncompletion', COL['head'], fs=8)
for y in [5.02, 3.98, 2.92, 1.88]:
    arrow(ax, 11.9, 3.77, 12.7, y)

box(ax, 6.2, 0.55, 4.2, 0.8, 'Training loss: NLL/quantile RSD + phase CE + deviation focal/BCE\n+ calibration + optional workflow-posterior supervision', COL['loss'], fs=8)
arrow(ax, 13.55, 1.55, 10.4, 1.0)

fig.tight_layout()
fig.savefig(FIG_DIR / 'cw_bariatricrsd_architecture.png', bbox_inches='tight')
plt.show()


## 4. Why This Model Can Win

The new model has three defensible advantages:

1. **It matches deployment.** At minute 20, it only sees what has happened by minute 20.
2. **It learns workflow uncertainty.** Early in the case, multiple operative styles may be plausible; later, the posterior sharpens.
3. **It separates causal inference from oracle analysis.** The full-video phase-order cluster becomes an upper bound, not the main method.

This gives reviewers a clean story: the method improves RSD by estimating and using the evolving workflow state.


In [ ]:
components = pd.DataFrame([
    ('Visual encoder', 'Frame-level surgical representation', 'Use ViT-B/ImageNet first; HecVL or surgical VLM is optional pretraining, not the core claim.'),
    ('Causal temporal encoder', 'Represent observed prefix only', 'Use causal mask or last-token pooling; no future frames for target time t.'),
    ('Workflow posterior q(z|prefix)', 'Estimate style/order from observed prefix', 'Can be trained against full-video cluster only as a label for prefix inference, not as an input.'),
    ('Soft style token', 'Condition the temporal state', 'Use expected embedding over q(z), not a hard oracle cluster.'),
    ('RSD distribution head', 'Predict remaining minutes + uncertainty', 'Report MAE plus calibration; distributional output is more publishable than scalar MSE.'),
    ('Deviation head', 'Rare event detection', 'Use focal loss or class-balanced BCE; report PR-AUC and event-level metrics.'),
    ('Phase head', 'Auxiliary workflow supervision', 'Use current-frame phase only; it improves representation and supports workflow posterior.'),
], columns=['Component', 'Role', 'Publication-safe framing'])
components


## 5. Workflow Posterior Instead of Oracle Cluster

Let `z` be a latent workflow style/order cluster. The model estimates:

```text
q_t = q(z | observed prefix up to time t)
s_t = sum_k q_t[k] * E[k]
```

Then `s_t` conditions prediction at time `t`.

Training can use the full-video cluster as a target for q(z|prefix), because that target is a label. The important part is that the **input** to q uses only the prefix.


In [ ]:
minutes = np.linspace(0, 120, 121)
centers = np.array([35, 55, 75, 95, 110, 130])
logits = []
for c in centers:
    logits.append(-0.0025 * (minutes - c) ** 2)
logits = np.vstack(logits).T
temp = np.linspace(3.5, 0.9, len(minutes))[:, None]
probs = np.exp(logits / temp)
probs = probs / probs.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 4.8))
for k in range(probs.shape[1]):
    ax.plot(minutes, probs[:, k], label=f'style {k}', linewidth=2)
ax.set_title('Desired Behavior: Prefix Workflow Posterior Sharpens Over Time')
ax.set_xlabel('Elapsed time (min)')
ax.set_ylabel('q(z | prefix)')
ax.set_ylim(0, 1)
ax.legend(ncol=3, fontsize=8)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / 'workflow_posterior_over_time.png', bbox_inches='tight')
plt.show()


## 6. Causal Prediction Setup

For every video, create training examples at multiple timestamps.

For a target time `t`:

- input frames: only times `<= t`
- target RSD: `duration - t`
- target phase: phase at `t`
- target deviation: event status at `t`, or event onset within a short future horizon if explicitly framed as forecasting
- workflow target: full-video style cluster can supervise q(z|prefix), but cannot be directly fed as an input

This fixes the main leakage problem in the current notebook.


In [ ]:
leakage_audit = pd.DataFrame([
    ('Full-video phase-order cluster as direct input', 'Leakage risk', 'Use only as oracle upper bound. Main model must infer q(z|prefix).'),
    ('Clip frames after target time t', 'Leakage for online RSD', 'Use causal windows ending at t or prefix transformer with causal mask.'),
    ('Middle-frame label with symmetric clip', 'Weak for deployment', 'Predict last/current frame label from past frames only.'),
    ('Validation conversion using true total duration', 'Metric risk', 'Predict absolute minutes or compute MAE from known target seconds directly.'),
    ('Best epoch from one run', 'Evidence risk', 'Report 5-fold mean/std and paired ablations.'),
    ('Deviation frame F1 only', 'Clinical metric risk', 'Add PR-AUC, event-level recall, onset tolerance, false alarms/hour.'),
], columns=['Issue', 'Why reviewers may object', 'Publishable fix'])
leakage_audit


## 7. Loss Function

A publishable version should predict both accuracy and uncertainty.

Recommended loss:

```text
L = L_rsd_dist + lambda_phase L_phase + lambda_dev L_dev + lambda_workflow L_workflow + lambda_cal L_cal
```

Where:

- `L_rsd_dist`: Gaussian NLL, log-normal NLL, or quantile pinball loss for remaining minutes.
- `L_phase`: current phase cross-entropy.
- `L_dev`: focal loss or class-balanced BCE.
- `L_workflow`: KL/CE between q(z|prefix) and final workflow style label, used only as supervision.
- `L_cal`: optional calibration penalty for RSD interval coverage.

Do not make negative total loss a central story. Reviewers care about calibrated performance, not whether learned log variances can go negative.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('Recommended Training Objective', fontsize=13)

terms = [
    ('RSD NLL /\nquantile loss', 0.6, 3.1, COL['head']),
    ('Phase CE', 2.7, 3.1, COL['head']),
    ('Deviation focal\n/ balanced BCE', 4.4, 3.1, COL['head']),
    ('Workflow posterior\nCE/KL', 6.7, 3.1, COL['workflow']),
    ('Calibration\npenalty', 8.5, 3.1, COL['loss']),
]
for label, x, y, color in terms:
    box(ax, x, y, 1.25, 0.85, label, color, fs=8)
    arrow(ax, x + 0.62, y, 5.0, 1.9)
box(ax, 4.0, 1.0, 2.1, 0.9, 'Total loss\nfor causal prefix examples', '#e5e7eb', fs=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'recommended_loss.png', bbox_inches='tight')
plt.show()


## 8. Experiments Needed for a Publishable Paper

The paper should be built around a ladder of ablations. Each row should use the same folds, same seeds, same encoder, and same training budget.


In [ ]:
experiment_ladder = pd.DataFrame([
    ('B0', 'Elapsed-time median baseline', 'No images; center/procedure duration prior only', 'Shows the non-visual floor.'),
    ('B1', 'ViT + causal temporal encoder', 'No phase-order or workflow token', 'Core visual-temporal baseline.'),
    ('B2', 'B1 + phase auxiliary head', 'Adds multi-task phase learning', 'Tests whether phase supervision helps RSD.'),
    ('B3', 'B2 + hard observed metadata token', 'Center/surgeon/procedure metadata only if known pre-op', 'Separates metadata benefit from learned workflow inference.'),
    ('M1', 'CW-BariatricRSD', 'Prefix-inferred workflow posterior soft token', 'Main method.'),
    ('M2', 'Oracle phase-order token', 'Full-video cluster input', 'Upper bound only; must be labeled oracle.'),
    ('M3', 'Random/shuffled workflow token', 'Wrong q(z)', 'Sanity check: token should not help when semantics are destroyed.'),
    ('X1', 'Cross-center train Bern -> val/test Strasbourg', 'Main domain shift test', 'Shows whether workflow conditioning helps generalization.'),
    ('X2', 'Train Strasbourg -> test Bern', 'Reverse domain shift', 'Rules out one-direction-only results.'),
], columns=['ID', 'Experiment', 'Definition', 'Why it matters'])
experiment_ladder


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('Evidence Ladder for Publication', fontsize=13)

items = [
    ('B0\nDuration prior', 0.4, 2.8, '#e5e7eb'),
    ('B1\nCausal ViT+HTA', 2.0, 2.8, COL['temporal']),
    ('B2\n+ phase aux', 3.6, 2.8, COL['head']),
    ('M1\n+ prefix workflow\nposterior', 5.2, 2.8, COL['workflow']),
    ('M2\nOracle cluster\nupper bound', 7.2, 2.8, COL['bad']),
]
for label, x, y, color in items:
    box(ax, x, y, 1.25, 0.95, label, color, fs=8)
for i in range(len(items) - 1):
    arrow(ax, items[i][1] + 1.25, 3.28, items[i+1][1], 3.28)
box(ax, 2.0, 0.9, 2.3, 0.7, '5-fold paired results\nmean +/- std', COL['good'], fs=8)
box(ax, 4.8, 0.9, 2.3, 0.7, 'Cross-center transfer\nBern <-> Strasbourg', COL['good'], fs=8)
box(ax, 7.6, 0.9, 1.6, 0.7, 'Calibration\nplots', COL['good'], fs=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'evidence_ladder.png', bbox_inches='tight')
plt.show()


## 9. Metrics to Report

RSD:

- MAE in minutes
- median absolute error
- MAE by elapsed quartile: early, middle, late surgery
- calibration: interval coverage if predicting uncertainty
- cross-center MAE

Deviation:

- PR-AUC, not just F1
- event-level recall with onset tolerance
- false alarms per hour
- per-phase recall

Phase:

- accuracy / balanced accuracy
- edit score or segmental F1 if doing phase sequence quality

Workflow posterior:

- prefix cluster accuracy over time
- entropy over time
- whether q(z|prefix) improves RSD when used as conditioning


## 10. Current Runs: What They Prove and What They Do Not

The local logs are useful, but they are not yet enough for a publication claim. They show that the idea is worth pursuing, not that the final method is proven.


In [ ]:
VAL_RE = re.compile(r'Epoch\s+(\d+)\s+\|\s+val_mae=([0-9.]+)min\s+\|\s+pearson_r=([\-0-9.]+)\s+\|\s+dev_f1=([\-0-9.]+)\s+\|\s+best=([0-9.]+)min')
RUN_RE = re.compile(r'Syncing run\s+([^\s]+)')
rows = []
for path in sorted(LOG_DIR.glob('*.log')):
    text = path.read_text(errors='replace')
    run_match = RUN_RE.search(text)
    run = run_match.group(1) if run_match else path.stem
    vals = []
    for m in VAL_RE.finditer(text):
        vals.append({
            'run': run,
            'log': path.name,
            'epoch': int(m.group(1)),
            'val_mae_min': float(m.group(2)),
            'pearson_r': float(m.group(3)),
            'dev_f1': float(m.group(4)),
            'best_mae_min_logged': float(m.group(5)),
        })
    if vals:
        best_mae = min(vals, key=lambda r: r['val_mae_min'])
        best_f1 = max(vals, key=lambda r: r['dev_f1'])
        rows.append({
            'run': run,
            'epochs_logged': len(vals),
            'best_mae_min': best_mae['val_mae_min'],
            'best_mae_epoch': best_mae['epoch'],
            'best_dev_f1': best_f1['dev_f1'],
            'best_f1_epoch': best_f1['epoch'],
            'final_mae_min': vals[-1]['val_mae_min'],
            'final_dev_f1': vals[-1]['dev_f1'],
        })
run_summary = pd.DataFrame(rows).sort_values('best_mae_min')
run_summary


In [ ]:
if not run_summary.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    plot_df = run_summary.copy()
    short = plot_df['run'].str.replace('run00', 'r', regex=False).str.replace('_mb140_', '\n', regex=False)
    axes[0].bar(range(len(plot_df)), plot_df['best_mae_min'], color='#a78bfa', edgecolor='black')
    axes[0].set_xticks(range(len(plot_df)))
    axes[0].set_xticklabels(short, rotation=45, ha='right', fontsize=8)
    axes[0].set_ylabel('Best val MAE (min)')
    axes[0].set_title('Current runs: RSD MAE')
    axes[0].grid(alpha=0.25, axis='y')

    axes[1].bar(range(len(plot_df)), plot_df['best_dev_f1'], color='#fca5a5', edgecolor='black')
    axes[1].set_xticks(range(len(plot_df)))
    axes[1].set_xticklabels(short, rotation=45, ha='right', fontsize=8)
    axes[1].set_ylabel('Best deviation F1')
    axes[1].set_title('Current runs: deviation F1')
    axes[1].grid(alpha=0.25, axis='y')

    fig.suptitle('Use current runs as pilot evidence only; rerun paired causal ablations for paper', y=1.03)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'current_runs_pilot_evidence.png', bbox_inches='tight')
    plt.show()


## 11. Interpretation of Existing Results

Based on the local logs:

- The best RSD MAE is from the k-means phase-order run, which is promising.
- Deviation F1 does not cleanly improve with the phase-order token.
- Cross-center performance is much worse, which is expected and useful for motivating the problem.
- The current setup is not yet leakage-safe enough to claim online performance.

Do not write: "Run 006 proves phase-order conditioning works."

Write instead: "Pilot results motivate a leakage-safe causal workflow-conditioning study."


## 12. Concrete Code Changes Needed

To make the model match the publishable idea, update the training pipeline like this:

1. Change sampling target from middle frame to **last/current frame** in the clip.
2. Ensure frame indices are all `<= t` for the target timestamp.
3. Add `WorkflowPosteriorNet` that consumes prefix phase logits/features and elapsed time.
4. Replace direct `phase_order_cluster` input in the main model with `q(z|prefix)`.
5. Keep direct `phase_order_cluster` only in an `oracle=True` experiment.
6. Predict absolute remaining minutes or a distribution over remaining minutes.
7. Add a duration-prior baseline and cross-center paired ablations.

These changes are more important than adding architectural complexity.


In [ ]:
pseudocode = """
# causal sample at time t
frames = sample_frames(video, end_time=t, lookback=L, stride=s)  # all frames <= t
h = visual_encoder(frames)
h_t = causal_temporal_encoder(h)[:, -1]

# prefix workflow state, no future labels
phase_prefix = phase_head_prefix(h.detach() or h)
workflow_features = build_prefix_features(phase_prefix, elapsed=t, metadata=m)
q_z = workflow_posterior(workflow_features)          # [B, K]
style = q_z @ workflow_embedding.weight              # soft token [B, D]

fused = fuse(h_t, style)
rsd_dist = rsd_distribution_head(fused)              # mean + scale or quantiles
dev_logit = deviation_head(fused)
phase_logit = phase_head(fused)

loss = rsd_nll(rsd_dist, rsd_minutes_target) \
     + lambda_dev * focal_bce(dev_logit, dev_target) \
     + lambda_phase * ce(phase_logit, phase_target) \
     + lambda_workflow * ce(q_z, final_style_cluster_label)
"""
print(pseudocode)


## 13. Paper Narrative That Can Work

Possible title:

**Causal Workflow-Conditioned Prediction of Remaining Duration and Deviations in Roux-en-Y Gastric Bypass Surgery**

Core abstract claim:

> Surgical duration prediction is difficult because procedure progress depends not only on visual state but also on evolving workflow style. We propose a causal workflow-conditioned temporal model that infers a posterior over operative styles from the observed prefix and uses this posterior to condition remaining-duration and deviation predictions. On MultiBypass140, we evaluate the method under within-center and cross-center splits, compare against duration priors, causal temporal transformers, and oracle workflow-conditioning upper bounds, and show improved early-case RSD accuracy and calibrated uncertainty under workflow variation.

This is much stronger than claiming novelty for a generic Transformer.


## 14. What Not To Claim

Avoid these claims unless the code and experiments truly support them:

- "The model knows surgeon style at inference" if style is computed from full labels.
- "Surgformer HTA" if the implementation is a simplified short/full attention block.
- "Operation log is a model contribution" unless evaluated clinically.
- "Phase-order token proves causality" without paired causal ablations.
- "Deviation detection solved" if only frame-level F1 is reported.

Reviewers are usually forgiving about incremental architecture if the experimental design is rigorous. They are not forgiving about leakage.


## 15. Success Criteria Before Submission

Minimum evidence package:

- 5-fold within-center results with mean/std.
- Cross-center Bern -> Strasbourg and Strasbourg -> Bern.
- Causal no-workflow baseline vs causal prefix-workflow model.
- Oracle cluster upper bound clearly marked as non-deployable.
- RSD MAE by elapsed quartile, especially early surgery.
- Calibration curve for predicted RSD intervals.
- Deviation PR-AUC and event-level detection.
- Ablation where workflow tokens are shuffled/randomized.

If these are positive, the paper has a credible contribution.


In [ ]:
checklist = pd.DataFrame([
    ('Causal dataset sampling', 'Required', 'Not optional. This fixes leakage.'),
    ('Prefix workflow posterior', 'Required', 'Main method.'),
    ('Oracle phase-order upper bound', 'Recommended', 'Useful but must be labeled oracle.'),
    ('Duration-prior baseline', 'Required', 'RSD can be deceptively strong with priors.'),
    ('5-fold paired ablations', 'Required', 'Needed for credibility.'),
    ('Cross-center transfer', 'Required', 'Best motivation for workflow conditioning.'),
    ('Calibration analysis', 'Strongly recommended', 'Makes RSD clinically more useful.'),
    ('Event-level deviation metrics', 'Strongly recommended', 'Frame F1 is not enough.'),
], columns=['Item', 'Priority', 'Reason'])
checklist


## 16. Bottom Line

The model can become publishable if the contribution is reframed as **causal workflow-state conditioning**, not as direct full-video phase-order conditioning.

The clean final story:

1. Surgical duration varies by workflow style.
2. Workflow style is uncertain early and becomes clearer over time.
3. A model can infer that uncertainty from the observed prefix.
4. Conditioning RSD/deviation prediction on that inferred workflow state improves performance and calibration, especially under cross-center workflow shift.

That is the idea to build and test.


In [ ]:
print(f'Notebook figures written to: {FIG_DIR}')
print('Primary figures:')
for p in sorted(FIG_DIR.glob('*.png')):
    print('-', p.relative_to(PROJECT))
